In [1]:
from ultralytics import YOLO
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd

### Load the pose estimator

In [2]:
pose = YOLO('yolo11m-pose.pt')

### Prepare DataFrame for data analysis

In [3]:
current_path = Path.cwd()
dataset_path = Path(os.path.join(current_path, 'datasets/fall_not_fall'))
dataset_path.exists()

True

In [23]:
def get_points(image_path, model, conf=0.8):
    results = model.predict(image_path, conf=conf, imgsz=640, verbose=False)
    keypoints = results[0].keypoints.xy.cpu().numpy()
    if len(keypoints) > 0:
        bboxes = results[0].boxes.xyxy[0].cpu().numpy().astype(int)
    else:
        bboxes = []

    return keypoints, bboxes

In [68]:
def nose_ref_point(point, nose):
    new_point = (float(point[0] - nose[0]), float(point[1] - nose[1]))
    return tuple(new_point)

In [89]:
def get_datapoints(dataset_path):
    dir_lst = ['train', 'test', 'valid']
    features, targets = [], []
    values = ['not fall', 'fall']

    for lst in dir_lst:
        path = Path(os.path.join(dataset_path, lst))
        print(f'Prepared for {path}')
        if path.exists:
            # Check for valid image extension
            img_ext = ['*.jpg', '*.jpeg', '*.png']
            images_path = []
            for ext in img_ext:
                images_path.extend(Path(os.path.join(path, 'images')).glob(ext))

            # Search for image annotation path
            for image_path in tqdm(images_path, desc='Processing image...'):
                annot_path = Path(os.path.join(path, 'labels', f'{image_path.stem}.txt'))
                
                if annot_path.exists():
                    keypoints, bboxes = get_points(image_path=image_path, model=pose)
                    
                    lines = []
                    with open(annot_path, 'r') as file:
                        lines = file.readlines()

                    if keypoints.shape[0] != len(lines):
                        print(f'Problem occured with file "{annot_path}"')
                        print(keypoints.shape, len(lines))
                        continue
                    
                    for points, line in zip(keypoints, lines):
                        nose = points[0]
                        new_point = list(map(lambda p: nose_ref_point(p, nose), points))
                        new_point.append(bboxes[:2])
                        new_point.append(bboxes[2:])
                        features.append(new_point)
                        targets.append(values[int(line.strip().split(' ')[0])])

    # Convert list to numpy array
    # features = np.array(features)
    # targets = np.array(targets)

    return features, targets

In [90]:
features, targets = get_datapoints(dataset_path=dataset_path)
# features.shape, targets.shape

Prepared for /home/labuser/Joy/IntellgiCare/datasets/fall_not_fall/train


Processing image...:  46%|████▌     | 85/185 [00:02<00:03, 32.91it/s]

Problem occured with file "/home/labuser/Joy/IntellgiCare/datasets/fall_not_fall/train/labels/IMG_20220908_215739_jpg.rf.8f75b953299f805bd899a3f1ca57ae5e.txt"
(0, 17, 2) 1


Processing image...:  82%|████████▏ | 152/185 [00:04<00:00, 37.16it/s]

Problem occured with file "/home/labuser/Joy/IntellgiCare/datasets/fall_not_fall/train/labels/person-63-_jpg.rf.052fbf07d4f712aed5cd94405e2a53c5.txt"
(1, 17, 2) 2


Processing image...: 100%|██████████| 185/185 [00:05<00:00, 35.44it/s]


Prepared for /home/labuser/Joy/IntellgiCare/datasets/fall_not_fall/test


Processing image...: 100%|██████████| 25/25 [00:00<00:00, 37.04it/s]


Prepared for /home/labuser/Joy/IntellgiCare/datasets/fall_not_fall/valid


Processing image...: 100%|██████████| 25/25 [00:00<00:00, 33.64it/s]


In [91]:
features

[[(0.0, 0.0),
  (2.969818115234375, -3.343017578125),
  (4.75787353515625, -4.371337890625),
  (16.017974853515625, 0.763031005859375),
  (24.252685546875, -3.33013916015625),
  (22.419921875, 26.064361572265625),
  (41.724853515625, 18.137359619140625),
  (19.771759033203125, 60.0574951171875),
  (51.76800537109375, 47.167877197265625),
  (5.402587890625, 86.25283813476562),
  (41.0611572265625, 67.84780883789062),
  (21.951141357421875, 83.23684692382812),
  (35.077728271484375, 76.83175659179688),
  (16.240264892578125, 122.56332397460938),
  (31.33636474609375, 111.39404296875),
  (20.480865478515625, 165.65145874023438),
  (37.42388916015625, 149.59182739257812),
  array([291, 289]),
  array([366, 492])],
 [(0.0, 0.0),
  (2.203216552734375, -5.118621826171875),
  (-5.887847900390625, -6.091644287109375),
  (-5.5640869140625, -3.91717529296875),
  (-26.0697021484375, -5.3277587890625),
  (-1.70098876953125, 21.91534423828125),
  (-49.882415771484375, 22.85565185546875),
  (5.635406

In [92]:
columns = [
    'Nose', 
    'Left_Eye', 
    'Right_Eye', 
    'Left_Ear', 
    'Right_Ear', 
    'Left_Shoulder', 
    'Right_Shoulder', 
    'Left_Elbow', 
    'Right_Elbow', 
    'Left_Wrist', 
    'Right_Wrist', 
    'Left_Hip', 
    'Right_Hip', 
    'Left_Knee', 
    'Right_Knee', 
    'Left_Ankle', 
    'Right_Ankle',
    'BBoxes_p1',
    'BBoxes_p2'
]

In [93]:
df = pd.concat([
    pd.DataFrame(features, columns=columns),
    pd.DataFrame(targets, columns=['targets'])
], axis=1)
df.head()

,Nose,Left_Eye,Right_Eye,Left_Ear,Right_Ear,Left_Shoulder,Right_Shoulder,Left_Elbow,Right_Elbow,Left_Wrist,Right_Wrist,Left_Hip,Right_Hip,Left_Knee,Right_Knee,Left_Ankle,Right_Ankle,BBoxes_p1,BBoxes_p2,targets
0,"(0.0, 0.0)","(2.969818115234375, -3.343017578125)","(4.75787353515625, -4.371337890625)","(16.017974853515625, 0.763031005859375)","(24.252685546875, -3.33013916015625)","(22.419921875, 26.064361572265625)","(41.724853515625, 18.137359619140625)","(19.771759033203125, 60.0574951171875)","(51.76800537109375, 47.167877197265625)","(5.402587890625, 86.25283813476562)","(41.0611572265625, 67.84780883789062)","(21.951141357421875, 83.23684692382812)","(35.077728271484375, 76.83175659179688)","(16.240264892578125, 122.56332397460938)","(31.33636474609375, 111.39404296875)","(20.480865478515625, 165.65145874023438)","(37.42388916015625, 149.59182739257812)","[291, 289]","[366, 492]",not fall
1,"(0.0, 0.0)","(2.203216552734375, -5.118621826171875)","(-5.887847900390625, -6.091644287109375)","(-5.5640869140625, -3.91717529296875)","(-26.0697021484375, -5.3277587890625)","(-1.70098876953125, 21.91534423828125)","(-49.882415771484375, 22.85565185546875)","(5.635406494140625, 59.436004638671875)","(-70.62771606445312, 64.62896728515625)","(7.100677490234375, 90.60125732421875)","(-67.525634765625, 100.11004638671875)","(-15.848541259765625, 95.57498168945312)","(-46.81585693359375, 96.55389404296875)","(-20.456024169921875, 138.37966918945312)","(-58.841339111328125, 140.6790771484375)","(-32.686309814453125, 178.5489501953125)","(-61.57403564453125, 189.4188232421875)","[249, 312]","[347, 555]",not fall
2,"(0.0, 0.0)","(-0.56451416015625, -7.63092041015625)","(-8.668426513671875, -8.126861572265625)","(-25.22955322265625, -3.575927734375)","(-40.944427490234375, -3.416046142578125)","(-30.344024658203125, 41.03179931640625)","(-65.21206665039062, 44.718902587890625)","(-17.309722900390625, 111.17706298828125)","(-71.0257568359375, 115.06954956054688)","(-11.33380126953125, 169.08639526367188)","(-34.4765625, 172.41000366210938)","(-30.401611328125, 170.22705078125)","(-52.78472900390625, 174.1461181640625)","(-52.381683349609375, 242.91421508789062)","(-61.906036376953125, 248.68038940429688)","(-58.222900390625, 299.1534423828125)","(-67.9521484375, 316.8498229980469)","[232, 169]","[347, 561]",not fall
3,"(0.0, 0.0)","(12.612579345703125, -6.0320587158203125)","(-10.45867919921875, -7.76226806640625)","(25.917266845703125, 8.030670166015625)","(-28.996856689453125, 3.712005615234375)","(46.2015380859375, 54.033538818359375)","(-58.212921142578125, 49.78639221191406)","(71.405517578125, 107.01434326171875)","(-93.59994506835938, 99.63003540039062)","(72.35000610351562, 155.06396484375)","(-95.0213623046875, 157.5733642578125)","(21.578704833984375, 165.56607055664062)","(-45.820343017578125, 164.13714599609375)","(31.1717529296875, 230.83670043945312)","(-65.32342529296875, 231.77801513671875)","(30.572509765625, 285.08770751953125)","(-58.79730224609375, 291.7757263183594)","[195, 154]","[398, 518]",not fall
4,"(0.0, 0.0)","(4.9658203125, -3.471710205078125)","(5.11431884765625, -5.28509521484375)","(26.09405517578125, 2.82830810546875)","(23.7957763671875, -4.234710693359375)","(39.92218017578125, 34.65142822265625)","(38.3875732421875, 21.74945068359375)","(37.54119873046875, 75.71722412109375)","(29.748046875, 51.44708251953125)","(22.300140380859375, 106.97628784179688)","(12.896728515625, 76.3055419921875)","(29.7509765625, 101.79486083984375)","(29.24359130859375, 92.95401000976562)","(27.943695068359375, 141.6373291015625)","(26.30059814453125, 128.94952392578125)","(27.743408203125, 180.53558349609375)","(21.303436279296875, 164.8646240234375)","[294, 258]","[382, 486]",not fall


In [94]:
df.tail()

,Nose,Left_Eye,Right_Eye,Left_Ear,Right_Ear,Left_Shoulder,Right_Shoulder,Left_Elbow,Right_Elbow,Left_Wrist,Right_Wrist,Left_Hip,Right_Hip,Left_Knee,Right_Knee,Left_Ankle,Right_Ankle,BBoxes_p1,BBoxes_p2,targets
228,"(0.0, 0.0)","(-5.749755859375, -1.284637451171875)","(13.784759521484375, 3.816986083984375)","(-8.40185546875, 3.57757568359375)","(35.50726318359375, 12.13275146484375)","(-27.34088134765625, 22.230224609375)","(56.21514892578125, 28.79730224609375)","(-87.76412963867188, -9.3553466796875)","(59.86541748046875, -2.035919189453125)","(-147.61801147460938, -83.32843017578125)","(-36.50079345703125, -63.258026123046875)","(-24.02630615234375, -44.46331787109375)","(21.192657470703125, -39.4130859375)","(-51.05255126953125, -178.1220703125)","(-36.445648193359375, -157.38949584960938)","(-51.650970458984375, -278.28338623046875)","(-63.624664306640625, -266.5418701171875)","[163, 154]","[487, 592]",fall
229,"(0.0, 0.0)","(5.192352294921875, -2.391265869140625)","(15.23968505859375, -0.267822265625)","(49.82049560546875, 4.708526611328125)","(62.540283203125, 8.045501708984375)","(71.17962646484375, -0.784393310546875)","(98.46978759765625, 28.548492431640625)","(-38.24949645996094, -35.76898193359375)","(63.20208740234375, 42.0836181640625)","(-182.66366577148438, -96.23089599609375)","(-109.55949401855469, 23.068695068359375)","(67.86962890625, -31.7064208984375)","(75.2239990234375, -29.018310546875)","(-35.944366455078125, -63.142974853515625)","(-49.65907287597656, -33.652435302734375)","(-147.2527618408203, -135.72232055664062)","(-161.43301391601562, -112.66824340820312)","[22, 280]","[467, 540]",fall
230,"(0.0, 0.0)","(1.5728759765625, 0.986053466796875)","(6.92340087890625, -19.03179931640625)","(8.415863037109375, 5.2123870849609375)","(25.310577392578125, -38.49176025390625)","(53.449310302734375, 63.515472412109375)","(64.34170532226562, -43.345001220703125)","(77.80947875976562, 119.38540649414062)","(54.73321533203125, -53.38720703125)","(31.05419921875, 75.2777099609375)","(4.881378173828125, 8.3724365234375)","(184.97296142578125, 71.33126831054688)","(197.62893676757812, 21.5780029296875)","(164.04986572265625, 1.8660125732421875)","(192.84521484375, -100.01736450195312)","(235.43560791015625, 42.228271484375)","(256.8013916015625, -43.72996520996094)","[195, 128]","[569, 378]",fall
231,"(0.0, 0.0)","(-6.1801605224609375, -4.9896240234375)","(-4.0653533935546875, -4.75469970703125)","(-9.917449951171875, -9.6912841796875)","(-2.5818023681640625, -5.7222900390625)","(28.845123291015625, -1.377593994140625)","(38.52912902832031, -22.070281982421875)","(59.89129638671875, -43.67974853515625)","(71.59310913085938, -87.43264770507812)","(49.77642822265625, -69.6981201171875)","(63.944793701171875, -86.42950439453125)","(168.92625427246094, -36.3680419921875)","(173.6680145263672, -43.174102783203125)","(219.5813751220703, -124.91424560546875)","(220.5472869873047, -170.71737670898438)","(289.5345458984375, -77.79312133789062)","(282.605712890625, -106.67826843261719)","[13, 191]","[443, 468]",fall
232,"(0.0, 0.0)","(-3.929351806640625, 1.806671142578125)","(5.281982421875, -13.4349365234375)","(-29.427093505859375, 26.517974853515625)","(9.601776123046875, -6.44927978515625)","(-10.222503662109375, 93.81375122070312)","(54.72747802734375, 30.580322265625)","(16.930450439453125, 181.2559814453125)","(97.77902221679688, 91.35287475585938)","(10.78106689453125, 178.57479858398438)","(79.3013916015625, 148.79910278320312)","(114.61990356445312, 161.34893798828125)","(165.3359375, 133.55648803710938)","(130.73550415039062, 99.47372436523438)","(198.9720458984375, 38.541229248046875)","(196.53912353515625, 132.70016479492188)","(252.43402099609375, 82.23806762695312)","[110, 227]","[464, 505]",fall


In [95]:
## Save the data
df.to_csv('fall_not_fall_with_coordinate.csv', index=False)

### Visualize the dataset

In [14]:
import seaborn as sns
# sns.pairplot(df, hue='targets')